# Demo — Análisis de churn en un banco europeo
### Maestría en Fintech · ITBA · 2026

---

**Dataset:** 10.000 clientes de un banco europeo  
**Fuente:** [Kaggle — Churn for Bank Customers](https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers)  
**Pregunta:** ¿Qué perfil de cliente tiene mayor probabilidad de irse del banco?

Este notebook es una demo completa del tipo de análisis que vas a poder hacer al final del curso.  
No hace falta entender el código todavía — enfocate en lo que muestran los gráficos y en las conclusiones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Librerías listas.')

---
## 1. Cargar el dataset

Los datos viven en la nube. Una línea alcanza para traerlos.

In [ ]:
URL = 'https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv'

df = pd.read_csv(URL)

print(f'Filas:    {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
df.head()

---
## 2. ¿Qué tenemos?

Antes de cualquier análisis, entender el dataset.

In [ ]:
df.info()

In [ ]:
# Valores nulos por columna
nulos = df.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0] if nulos.any() else 'Ninguno — dataset limpio.')

In [ ]:
# Estadísticas descriptivas
df.describe().round(1)

In [ ]:
# Tasa de churn global
tasa = df['Churn'].mean()
print(f'Clientes que se fueron: {df["Churn"].sum():,} de {len(df):,} ({tasa:.1%})')

---
## 3. ¿Cómo se distribuye la cartera?

Antes de buscar qué causa el churn, hay que entender a quiénes tenemos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribución de edades
axes[0].hist(df['Age'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de edades')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Clientes')

# Distribución de balance
axes[1].hist(df['Balance'], bins=25, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de balance')
axes[1].set_xlabel('Balance (€)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Clientes por país
geo = df['Geography'].value_counts()
axes[2].bar(geo.index, geo.values, color='steelblue', edgecolor='white')
axes[2].set_title('Clientes por país')
for i, v in enumerate(geo.values):
    axes[2].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. ¿Quiénes se van?

Ahora sí: segmentar el churn por distintas variables para encontrar el perfil de riesgo.

In [ ]:
# Churn por país y por género
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por país
churn_geo = df.groupby('Geography')['Churn'].mean().sort_values(ascending=False)
bars = axes[0].bar(churn_geo.index, churn_geo.values * 100, color='salmon', edgecolor='white')
axes[0].set_title('Tasa de churn por país', fontsize=13)
axes[0].set_ylabel('% de clientes que se fueron')
axes[0].set_ylim(0, churn_geo.max() * 130)
for bar, val in zip(bars, churn_geo.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1%}', ha='center', fontweight='bold')

# Por género
churn_gen = df.groupby('Gender')['Churn'].mean().sort_values(ascending=False)
bars2 = axes[1].bar(churn_gen.index, churn_gen.values * 100, color='salmon', edgecolor='white')
axes[1].set_title('Tasa de churn por género', fontsize=13)
axes[1].set_ylabel('% de clientes que se fueron')
axes[1].set_ylim(0, churn_gen.max() * 130)
for bar, val in zip(bars2, churn_gen.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Churn por franja etaria
df['franja_edad'] = pd.cut(df['Age'],
                           bins=[18, 30, 40, 50, 60, 100],
                           labels=['18-30', '31-40', '41-50', '51-60', '60+'])

churn_edad = df.groupby('franja_edad', observed=True)['Churn'].mean()

plt.figure(figsize=(9, 4))
bars = plt.bar(churn_edad.index.astype(str), churn_edad.values * 100,
               color='salmon', edgecolor='white')
plt.title('Tasa de churn por franja etaria', fontsize=13)
plt.ylabel('% de clientes que se fueron')
plt.xlabel('Franja etaria')
plt.ylim(0, churn_edad.max() * 130)
for bar, val in zip(bars, churn_edad.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Balance promedio: ¿quiénes se van vs quiénes se quedan?
balance_churn = df.groupby('Churn')['Balance'].mean()

labels = ['Se quedaron', 'Se fueron']
colors = ['steelblue', 'salmon']
plt.figure(figsize=(7, 4))
bars = plt.bar(labels, balance_churn.values, color=colors, edgecolor='white')
plt.title('Balance promedio: ¿se van los que tienen más?', fontsize=13)
plt.ylabel('Balance promedio (€)')
for bar, val in zip(bars, balance_churn.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'€{val:,.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Número de productos: ¿más productos = más fidelidad?
churn_prod = df.groupby('Num Of Products')['Churn'].mean()

plt.figure(figsize=(8, 4))
bars = plt.bar(churn_prod.index.astype(str), churn_prod.values * 100,
               color='salmon', edgecolor='white')
plt.title('Tasa de churn por cantidad de productos', fontsize=13)
plt.xlabel('Número de productos')
plt.ylabel('% de clientes que se fueron')
for bar, val in zip(bars, churn_prod.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Resumen: el perfil del cliente que se va

Construir una tabla resumen para comunicar los hallazgos.

In [ ]:
resumen = df.groupby('Churn').agg(
    cantidad          = ('CustomerId', 'count'),
    edad_promedio     = ('Age', 'mean'),
    balance_promedio  = ('Balance', 'mean'),
    productos_prom    = ('Num Of Products', 'mean'),
    pct_activos       = ('Is Active Member', 'mean')
).round(1)

resumen.index = ['Se quedaron', 'Se fueron']
resumen['balance_promedio'] = resumen['balance_promedio'].apply(lambda x: f'€{x:,.0f}')
resumen['pct_activos']      = resumen['pct_activos'].apply(lambda x: f'{x:.0%}')
resumen

---
## Conclusiones

Con este análisis podemos armar un mensaje concreto para el equipo de retención:

1. **El problema tiene escala:** 1 de cada 5 clientes se fue — una tasa que justifica una campaña de retención dedicada.

2. **La geografía importa:** Alemania tiene una tasa de churn significativamente mayor que Francia y España. Vale la pena entender qué está pasando en ese mercado.

3. **La edad es el factor más discriminante:** Los clientes de 41 a 60 años se van más que los jóvenes. Puede reflejar una propuesta de valor que no está llegando al segmento de mediana edad.

4. **Las mujeres se van más que los hombres** — una señal para revisar si el producto o la comunicación está teniendo un sesgo de género.

5. **Tener 3 o 4 productos es señal de alerta, no de fidelidad.** La hipótesis intuitiva (más productos = más fidelidad) no se cumple acá — requiere investigación adicional.

> *Esto es lo que hace un analista de datos: convertir números en preguntas que el negocio tiene que responder.*